In [1]:
import pandas as pd
import numpy as np
import ast

movies = pd.read_csv('../data/tmdb_5000_movies.csv')
credits = pd.read_csv('../data/tmdb_5000_credits.csv')
credits.columns = ['movie_id', 'title', 'cast', 'crew']

df = movies.merge(credits, left_on='id', right_on='movie_id')
df = df.drop(columns=['title_y', 'movie_id'])
df = df.rename(columns={'title_x': 'title'})
df.shape

(4803, 22)

In [2]:
import ast

def get_list(x):
    L = ast.literal_eval(x)
    names = [i['name'] for i in L]
    return names

df['genres_list'] = df['genres'].apply(get_list)
df['keywords_list'] = df['keywords'].apply(get_list)
df['cast_list'] = df['cast'].apply(get_list)
df['cast_list'] = df['cast_list'].apply(lambda x: x[:3])

def get_director(x):
    crew = ast.literal_eval(x)
    for person in crew:
        if person['job'] == 'Director':
            return person['name']
    return np.nan

df['director'] = df['crew'].apply(get_director)
df[['title','genres_list','cast_list','director']].head()

,title,genres_list,cast_list,director
0,Avatar,"[Action, Adventure, Fantasy, Science Fiction]","[Sam Worthington, Zoe Saldana, Sigourney Weaver]",James Cameron
1,Pirates of the Caribbean: At World's End,"[Adventure, Fantasy, Action]","[Johnny Depp, Orlando Bloom, Keira Knightley]",Gore Verbinski
2,Spectre,"[Action, Adventure, Crime]","[Daniel Craig, Christoph Waltz, Léa Seydoux]",Sam Mendes
3,The Dark Knight Rises,"[Action, Crime, Drama, Thriller]","[Christian Bale, Michael Caine, Gary Oldman]",Christopher Nolan
4,John Carter,"[Action, Adventure, Science Fiction]","[Taylor Kitsch, Lynn Collins, Samantha Morton]",Andrew Stanton


In [3]:
def clean_names(x):
    if isinstance(x, list):
        return [str.lower(i.replace(" ", "")) for i in x]
    else:
        return ''

for col in ['genres_list', 'keywords_list', 'cast_list']:
    df[col] = df[col].apply(clean_names)

df['director_clean'] = df['director'].apply(lambda x: str.lower(x.replace(" ","")) if isinstance(x, str) else '')

df['tags'] = df['genres_list'] + df['keywords_list'] + df['cast_list']
df['tags'] = df['tags'].apply(lambda x: ' '.join(x)) + ' ' + df['director_clean']

from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(stop_words='english')
matrix = tfidf.fit_transform(df['tags'])
matrix.shape

(4803, 17832)

In [4]:
from sklearn.metrics.pairwise import cosine_similarity

sim = cosine_similarity(matrix)

def recommend(title):
    idx = df[df['title'] == title].index[0]
    scores = list(enumerate(sim[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    scores = scores[1:6]
    
    movies = []
    for i in scores:
        movies.append(df.iloc[i[0]]['title'])
    return movies

recommend('Avatar')

['Star Trek Into Darkness',
 'Aliens',
 'Alien³',
 'Predators',
 'Jupiter Ascending']

In [5]:
import numpy as np
np.random.seed(42)

n_users = 500
genres_all = df['genres_list'].explode().dropna().unique()

data = []
for u in range(n_users):
    fav = np.random.choice(genres_all, 2, replace=False)

    liked_movies = df[df['genres_list'].apply(lambda g: any(x in g for x in fav))]
    fav_picks = liked_movies.sample(min(14, len(liked_movies)))
    other_picks = df.drop(fav_picks.index).sample(6)
    picks = pd.concat([fav_picks, other_picks])

    for _, row in picks.iterrows():
        # users rate their favorite genres higher (around 4.5 vs. 3.0)
        liked = any(x in row['genres_list'] for x in fav)
        base = 4.5 if liked else 3.0
        r = int(np.clip(round(base + np.random.normal(0, 0.7)), 1, 5))
        data.append([u, row['id'], r])

ratings_df = pd.DataFrame(data, columns=['user_id', 'movie_id', 'rating'])
ratings_df.head()

,user_id,movie_id,rating
0,0,2320,5
1,0,10431,4
2,0,45138,5
3,0,1735,4
4,0,5174,5


In [6]:
from surprise import Dataset, Reader, SVD, NormalPredictor, BaselineOnly, accuracy
from surprise.model_selection import train_test_split

reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(ratings_df, reader)
train, test = train_test_split(data, test_size=0.2, random_state=42)

for name, algo in [("Random guess", NormalPredictor()),
                   ("Baseline (averages)", BaselineOnly(verbose=False)),
                   ("SVD", SVD(random_state=42))]:
    algo.fit(train)
    rmse = accuracy.rmse(algo.test(test), verbose=False)
    print(f"{name}: RMSE {rmse:.3f}")

model = SVD(random_state=42)
model.fit(train)

Random guess: RMSE 1.204
Baseline (averages): RMSE 0.910
SVD: RMSE 0.923


In [7]:
indices = pd.Series(df.index, index=df['title'])
indices = indices[~indices.index.duplicated()]

def hybrid(title, user_id, n=5, w=0.5):
    idx = indices[title]
    scores = list(enumerate(sim[idx]))
    
    results = []
    for i, content_score in scores:
        if i == idx:
            continue
        mid = df.iloc[i]['id']
        collab_score = model.predict(user_id, mid).est / 5
        final = w*content_score + (1-w)*collab_score
        results.append((i, final))
    
    results.sort(key=lambda x: x[1], reverse=True)
    top = results[:n]
    return [df.iloc[i]['title'] for i,_ in top]

print(hybrid('Avatar', 1))
print(hybrid('Avatar', 2))

['Aliens', 'Jupiter Ascending', 'Titan A.E.', "Ender's Game", 'Star Trek Into Darkness']
['Aliens', 'Star Trek Into Darkness', 'Jupiter Ascending', 'Planet 51', 'Titan A.E.']


In [8]:
df['ratio'] = df['revenue'] / df['budget']
df['hit'] = (df['ratio'] >= 2).astype(int)

d = df[(df['budget'] >= 1000) & (df['revenue'] > 0)].copy()
d['release_date'] = pd.to_datetime(d['release_date'], errors='coerce')
d['month'] = d['release_date'].dt.month
d['genre1'] = d['genres_list'].apply(lambda x: x[0] if len(x)>0 else 'Unknown')

dummies = pd.get_dummies(d['genre1'], prefix='g')
X = pd.concat([d[['budget','runtime','popularity','month']], dummies], axis=1)
X = X.fillna(X.median())
y = d['hit']

X.shape

(3216, 24)

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

rf = RandomForestClassifier(n_estimators=100, random_state=1)
rf.fit(X_train, y_train)

pred = rf.predict(X_test)
print(accuracy_score(y_test, pred))
print(classification_report(y_test, pred))

0.7003105590062112
              precision    recall  f1-score   support

           0       0.66      0.63      0.64       276
           1       0.73      0.76      0.74       368

    accuracy                           0.70       644
   macro avg       0.69      0.69      0.69       644
weighted avg       0.70      0.70      0.70       644



In [10]:
imp = pd.Series(rf.feature_importances_, index=X.columns)
imp.sort_values(ascending=False).head(10)

popularity     0.369363
budget         0.230756
runtime        0.185646
month          0.112399
g_action       0.012469
g_horror       0.012019
g_comedy       0.010905
g_drama        0.010432
g_adventure    0.007976
g_thriller     0.007617
dtype: float64

In [11]:
X2 = X.drop(columns=['popularity'])

X_train2, X_test2, y_train2, y_test2 = train_test_split(X2, y, test_size=0.2, random_state=1)

rf2 = RandomForestClassifier(n_estimators=100, random_state=1)
rf2.fit(X_train2, y_train2)

pred2 = rf2.predict(X_test2)
print(accuracy_score(y_test2, pred2))
print(classification_report(y_test2, pred2))

0.5450310559006211
              precision    recall  f1-score   support

           0       0.47      0.47      0.47       276
           1       0.60      0.60      0.60       368

    accuracy                           0.55       644
   macro avg       0.54      0.54      0.54       644
weighted avg       0.55      0.55      0.55       644



In [12]:
imp2 = pd.Series(rf2.feature_importances_, index=X2.columns)
imp2.sort_values(ascending=False).head(10)

budget         0.380257
runtime        0.347311
month          0.161586
g_horror       0.013801
g_drama        0.011546
g_action       0.010452
g_comedy       0.010342
g_adventure    0.009021
g_thriller     0.007664
g_crime        0.007238
dtype: float64